# 02 — Triage: apply the beach rule, drop blobs, pick sites to look atReads `output/regions_20ft.csv` and narrows it to a short list you can eyeball.Two filters do the work: Larissa's beach rule, and the total-acres fix carriedover from NC.## The beach rule, done honestlyLarissa's line is **30-mile drive** from the ocean. Drive distance is always atleast straight-line distance, so:- straight-line **> 30 mi** → drive is also > 30 → **fails, drop now.**- straight-line **≤ 30 mi** → drive *might* still exceed 30, because water  (Back Bay, the North Landing River) forces detours → **keep as candidate,**  confirm with a real drive-time at the finalist stage.So this notebook hard-drops only what's provably out, and never deletes apossible winner on an approximate number — the same discipline we used foraccess scoring in NC.## The metric, fixed (again)Region *count* is non-monotonic — a blob fragments as the threshold rises, sothe count can climb while real high ground shrinks. Total acres above thresholdis the honest measure. And where the threshold doesn't bite, regions merge intolocality-sized blobs; anything over a couple thousand acres is a landscape, nota parcel, and usually federal here (Great Dismal Swamp).

## 1 · Load

In [ ]:
from pathlib import Pathimport numpy as np, pandas as pdimport va_aoi as AOUT_DIR = Path("output")regions = pd.read_csv(OUT_DIR / "regions_20ft.csv")print(f"{len(regions)} regions from stage 1")if len(regions):    print(regions.locality.value_counts().to_string())

## 2 · Apply the beach rule

In [ ]:
# --- Apply the beach rule ---------------------------------------------------regions["beach_ok"] = regions.beach_mi <= A.BEACH_RULE_MIout = regions[~regions.beach_ok]keep = regions[regions.beach_ok].copy()print(f"{len(out)} regions dropped as > {A.BEACH_RULE_MI:g} mi straight-line "      f"(provably fail the drive rule)")print(f"{len(keep)} regions survive as candidates\n")if len(keep):    print(keep.groupby("locality")              .agg(n=("region_id","count"),                   nearest_mi=("beach_mi","min"),                   farthest_mi=("beach_mi","max"))              .to_string(float_format=lambda v: f"{v:.1f}"))

## 3 · Total-acres + drop blobs

In [ ]:
# --- Total-acres view + drop the blobs --------------------------------------acre_totals = (keep.groupby("locality")["acres"]               .agg(total_acres="sum", n="count", biggest="max")               .sort_values("total_acres", ascending=False))acre_totals["biggest_share"] = acre_totals.biggest / acre_totals.total_acresprint(acre_totals.to_string(float_format=lambda v: f"{v:,.1f}"))print("\n(biggest_share near 1.0 = one merged blob = 'this area is just high',")print(" not a site. Near 0 = genuinely scattered pockets.)")sites = keep[    keep.pad_fits    & keep.acres.between(A.MIN_SITE_ACRES, A.MAX_REGION_ACRES)].copy()print(f"\n{len(sites)} sites in the {A.MIN_SITE_ACRES:g}-{A.MAX_REGION_ACRES:g} "      f"acre window with a buildable pad (from {len(keep)} candidates)")

## 4 · Flag protected land

In [ ]:
# --- Flag protected / federal land ------------------------------------------# Great Dismal Swamp, Back Bay, False Cape, the naval fields. Approximate --# verify each boundary in the map before discarding. Big blobs get an extra flag.sites["exclusion"] = [A.flag_exclusion(lon, lat)                      for lon, lat in zip(sites.lon, sites.lat)]sites.loc[sites.acres > 2000, "exclusion"] = sites.loc[sites.acres > 2000, "exclusion"] \    .replace("", "large blob - likely public")n_flag = (sites.exclusion != "").sum()print(f"{n_flag} sites flagged (protected/federal/blob), {len(sites)-n_flag} clean")if n_flag:    print(sites[sites.exclusion != ""].exclusion.value_counts().to_string())

## 5 · Checklist

In [ ]:
# --- Satellite links + checklist --------------------------------------------sites["maps_url"] = sites.apply(    lambda r: f"https://www.google.com/maps/@{r.lat:.6f},{r.lon:.6f},1000m/data=!3m1!1e3",    axis=1)sites = sites.sort_values(["exclusion", "beach_mi", "pad_acres"],                          ascending=[True, True, False])cols = ["locality", "acres", "mean_ft", "max_ft", "pad_acres",        "beach_mi", "exclusion", "lat", "lon", "maps_url"]checklist = sites[cols].copy()checklist["looks_like"] = ""     # subdivision / pine / farm / clearchecklist["keep"] = ""           # y / nchecklist.to_csv(OUT_DIR / "site_checklist.csv", index=False)print(f"[ok] {len(checklist)} sites -> output/site_checklist.csv\n")show = checklist[checklist.exclusion == ""].drop(    columns=["maps_url", "looks_like", "keep"])print("Clean sites, nearest-beach first:")print(show.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Next**The eyeball pass.** Open `site_checklist.csv`, click each `maps_url`, fill in`looks_like` and `keep`. Given the terrain, expect this list to be short and toget shorter fast — a lot of "clear" high ground here will turn out to be activefarmland, which is fine (farmland is buyable and often already cleared for a pad).**Then, in order:**1. **Drive-time confirm.** For the `keep=y` survivors, a real 30-minute-drive   check against the oceanfront — this is where a few ≤30-mi-straight-line sites   drop out for being a 40-minute drive around the water.2. **Validate the elevation itself.** Check the banded map against where Isabel   (2003) and the 2009 Nor'Ida actually put water. Known-flooded ground should   fall in the reds and oranges. This is the Chaco Figure-11 check and it   happens before any realtor call.3. **Parcels** (notebook 04, now VGIN) → owners of the surviving high ground.4. **Listings** (notebook 05) → what's for sale, four sources, VA zips.5. **1 m lidar** on the finalists. Subsidence plus 10 m error means the last   couple of feet of margin need the high-resolution data before an offer.**Still worth sitting with:** the overlap of "close to the surf" and "safe fromsurge" is genuinely thin on this coast. If it comes back nearly empty, that'sreal information — it means the honest move is either Larissa's 30 becomes 35,or the high pad gets engineered rather than found. Elevation-first is what letsyou see that clearly instead of discovering it after an offer.